# Generating pharaoh-format data for James Cuenod

James is testing subword tokenization: to do this, he needs three files.

* Source and target texts, one line per verse, with punctuation split off.
* Pharaoh-format data with the same line correspondence.


In [11]:
# Setup

import pandas as pd
#pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

from src.burrito import DATAPATH, AlignmentSet, pharaoh
LANGDATAPATH = DATAPATH.parent.parent / "alignments-eng/data"
alset = AlignmentSet(targetlanguage="eng", sourceid="WLCM", targetid="BSB", langdatapath=LANGDATAPATH)
pm = pharaoh.PharaohMapper(alset)



        - sourcepath: /Users/sboisen/git/Clear-Bible/internal-Alignments/data/sources/WLCM.tsv
        - targetpath: /Users/sboisen/git/Clear-Bible/alignments-eng/data/targets/BSB/ot_BSB.tsv
        - alignmentpath: /Users/sboisen/git/Clear-Bible/alignments-eng/data/alignments/BSB/WLCM-BSB-manual.json
        - tomlpath: /Users/sboisen/git/Clear-Bible/alignments-eng/data/alignments/BSB/WLCM-BSB-manual.toml
        
Dropping 842 bad alignment records. Instances in self.malaligned.
Missing target token	838
No source	4


In [4]:
# Setup for writing data
EXPDATAPATH = DATAPATH.parent.parent / "alignments-eng/exp"
expdir = EXPDATAPATH / "20240726-James"
expdir.mkdir(parents=True, exist_ok=True)
# write files
pm.write_pharaoh(expdir / f"{alset.sourceid}-{alset.targetid}-pharaoh.txt")
pm.write_vline(expdir / f"{alset.sourceid}-vline.txt", typeattr="sources")
pm.write_vline(expdir / f"{alset.targetid}-vline.txt", typeattr="targets")


## Status

Created output, which seems reasonable barring other feedback. It's vref-indexed by source, so leaves blank lines in `target-vline.text` as necessary. 

A few verses have partial versification issues, like 04025019 where the WLCM content corresponds to "After the plague", which is just _the first part_ of 04026001. So the source_verse correspondence is *partial*, and therefore not captured in the `source_verse` attribute. That means it's valid to have alignment data for this verse even though there's no corresponding vref tokens. 

Verses like this:
```
04025019
09020042
11022043
13012004
19051001
19052001
19054001
19060001
```

### Viewing specific lines

* First find the position of a BCV value in source: `list(pm.bcv["sources"].keys()).index("04025019")`
```
>>> list(pm.bcv["sources"].keys()).index("04025019")
4490
```
* Then use this shell command for viewing a few lines around it:
```
--> head -n 4492 < BSB-vline.txt | tail -n 3
For they assailed you deceitfully when they seduced you in the matter of Peor and their sister Cozbi , the daughter of the Midianite leader , the woman who was killed on the day the plague came because of Peor . ”

After the plague had ended , the LORD said to Moses and Eleazar son of Aaron the priest ,
--> head -n 4492 < WLCM-vline.txt | tail -n 3
כִּי צֹרְרִים הֵם לָ כֶם בְּ נִכְלֵי הֶם אֲשֶׁר נִכְּלוּ לָ כֶם עַל דְּבַר פְּעוֹר וְ עַל דְּבַר כָּזְבִּי בַת נְשִׂיא מִדְיָן אֲחֹתָ ם הַ מֻּכָּה בְ יוֹם הַ מַּגֵּפָה עַל דְּבַר פְּעוֹר
וַ יְהִי אַחֲרֵי הַ מַּגֵּפָה
וַ יֹּאמֶר יְהוָה אֶל מֹשֶׁה וְ אֶל אֶלְעָזָר בֶּן אַהֲרֹן הַ כֹּהֵן לֵ אמֹר
```